In [10]:
import numpy as np
from skimage.feature import local_binary_pattern
from typing import Literal
import os
import glob
from pathlib import Path
import cv2
import tqdm

# Functions

In [2]:
def spatial_lbp_histogram(image: np.ndarray,
                          P: int = 8,
                          R: int = 2,
                          method: Literal['default', 'ror', 'uniform', 'nri_uniform', 'var'] = 'nri_uniform',
                          grid_x: int = 2,
                          grid_y: int = 2):
   
    assert isinstance(image, np.ndarray) and len(image.shape) == 2, "Input must be a 2D grayscale image."

    lbp_desc = local_binary_pattern(image, P, R, method=method)

    if method == 'uniform':
        n_bins = P + 2
    elif method == 'nri_uniform':
        n_bins = P * (P - 1) + 3
    else: # 'default', 'ror', 'var'
        n_bins = 2**P
        
    h, w = lbp_desc.shape
    cell_h, cell_w = h // grid_y, w // grid_x

    full_hist = []
    for y in range(grid_y):
        for x in range(grid_x):
            y_start, y_end = y * cell_h, (y + 1) * cell_h
            x_start, x_end = x * cell_w, (x + 1) * cell_w
            
            cell = lbp_desc[y_start:y_end, x_start:x_end]
            
            hist, _ = np.histogram(cell.ravel(),
                                   bins=n_bins,
                                   range=(0, n_bins),
                                   density=True)
            
            full_hist.append(hist)
    
    return np.concatenate(full_hist)

# Code

## Turn images into Gray

In [21]:
dataset_origin = Path("images")
dest_path = Path('gray_images')
dest_path.mkdir(exist_ok=True)

class_folders = [p for p in dataset_origin.iterdir() if p.is_dir()]

for class_folder in tqdm.tqdm(class_folders, desc="Processing Images"):
    class_name = class_folder.name

    dest_class_path = dest_path / class_name
    dest_class_path.mkdir(exist_ok=True)
    
    for image_path in glob.glob(os.path.join(class_folder, "*.jpg")):
        image = cv2.imread(str(image_path))
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image_path = Path(image_path)
        output_path = dest_class_path / image_path.name
        cv2.imwrite(str(output_path), gray_image)

Processing Images: 100%|██████████| 10/10 [00:04<00:00,  2.12it/s]


## Extract LBP and GLCM features

In [ ]:
dataset_origin = Path("gray_images")
dest_path = Path("features")
dest_path.mkdir(exist_ok=True)

features_df = pd.DataFrame()

class_folders = [p for p in dataset_origin.iterdir() if p.is_dir()]

for class_folder in tqdm.tqdm(class_folders, desc="Processing Images"):
    class_name = class_folder.name

    
    for image_path in glob.glob(os.path.join(class_folder, "*.jpg")):
        image = cv2.imread(str(image_path))
        features = spatial_lbp_histogram(image, grid_x=1, grid_y=1)
        if features_df.empty:
            features_df